# GRU4Rec Multi-Behavior Next-Item — FULL CATALOG

Mục tiêu:
- Dùng lịch sử tương tác user để dự đoán **item ở interaction kế tiếp**.
- Behavior là feature đầu vào, không phải label.
- **Không hard-cap catalog**: toàn bộ item trong train mapping đều có thể là candidate.
- Item embedding nằm trên **CPU** và dùng sparse gradients.
- GRU / event embedding / projection nằm trên GPU.
- Mỗi batch chỉ gather positive + sampled negative item vectors cần thiết.
- Inference build FAISS index theo chunk trên toàn catalog.

Pipeline:
1. Read MerRec schema
2. Build full item vocabulary
3. Build temporal user sequences
4. Train GRU4Rec next-item with sampled softmax
5. Evaluate Recall/NDCG/MRR
6. Build full-catalog FAISS index
7. `recommend_next(user_history)`

Quan trọng: catalog hàng chục triệu item khiến full item embedding vẫn rất lớn trong RAM.
Notebook sẽ in memory estimate trước khi train và dừng sớm nếu RAM khả dụng không đủ.


## Bản FAST + per-epoch metrics
- `batch_size=128`, `num_negatives=64`, tối đa `1,000,000` windows/epoch.
- In sampled validation sau mỗi epoch: Recall/NDCG/MRR @ 10/20/50/100.
- Sampled metric dùng lịch sử TRAIN + target VAL thật; không được gọi là full-catalog metric.
- Lưu latest checkpoint + ZIP sau mỗi epoch, epoch mới ghi đè epoch trước.
- Lưu thêm best checkpoint theo sampled Recall@20.


In [1]:
import os, gc, json, math, random, time
from pathlib import Path

import numpy as np
import polars as pl
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import faiss
except Exception:
    faiss = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)


DEVICE = cuda


In [2]:
CFG = {
    "seed": SEED,

    # Full catalog
    "item_embed_dim": 32,
    "event_embed_dim": 16,
    "hidden_dim": 128,
    "gru_layers": 2,
    "dropout": 0.2,
    "max_seq_len": 100,

    # Training
    "batch_size": 128,
    "grad_accum_steps": 1,
    "num_negatives": 64,
    "temperature": 0.10,
    "epochs": 8,
    "min_epochs": 3,
    "patience": 2,

    # Optimizers
    "item_lr": 0.05,      # sparse CPU embedding -> SGD
    "lr": 8e-4,           # dense GPU model -> AdamW
    "weight_decay": 1e-5,
    "grad_clip": 1.0,
    "amp": True,

    # Optional bound for Kaggle runtime. Set None for full epoch.
    "max_train_windows_per_epoch": 1_000_000,

    # Evaluation
    "val_users": 10000,
    "test_users": 10000,
    "eval_ks": [10, 20, 50, 100],
    "filter_seen_items": False,

    # Fast per-epoch sampled validation.
    # Đây KHÔNG phải full-catalog metric; target được xếp hạng cùng sampled negatives.
    "eval_every": 1,
    "eval_sampled_users": 2000,
    "eval_sampled_negatives": 255,
    "eval_batch_size": 128,

    # Không ép gc/empty_cache sau mọi row-group vì rất chậm.
    "cleanup_every_rowgroups": 20,

    # Tạo ZIP sau mỗi epoch (ghi đè ZIP cũ).
    "zip_each_epoch": True,

    # FAISS full-catalog build
    "item_vector_batch_size": 100000,
    "faiss_train_items": 500000,
    "faiss_nprobe": 64,
    "faiss_pq_m": 16,     # hidden_dim 128 divisible by 16
    "faiss_pq_nbits": 8,
}

EVENT_TO_IDX = {
    "unknown": 0,
    "view": 1,
    "like": 2,
    "cart": 3,
    "offer": 4,
    "buy_start": 5,
    "buy_comp": 6,
}
EVENT_VOCAB_SIZE = max(EVENT_TO_IDX.values()) + 1
PAD, OOV = 0, 1

print(json.dumps(CFG, indent=2))
print("Effective batch =", CFG["batch_size"] * CFG["grad_accum_steps"])


{
  "seed": 42,
  "item_embed_dim": 32,
  "event_embed_dim": 16,
  "hidden_dim": 128,
  "gru_layers": 2,
  "dropout": 0.2,
  "max_seq_len": 100,
  "batch_size": 128,
  "grad_accum_steps": 1,
  "num_negatives": 64,
  "temperature": 0.1,
  "epochs": 8,
  "min_epochs": 3,
  "patience": 2,
  "item_lr": 0.05,
  "lr": 0.0008,
  "weight_decay": 1e-05,
  "grad_clip": 1.0,
  "amp": true,
  "max_train_windows_per_epoch": 1000000,
  "val_users": 10000,
  "test_users": 10000,
  "eval_ks": [
    10,
    20,
    50,
    100
  ],
  "filter_seen_items": false,
  "eval_every": 1,
  "eval_sampled_users": 2000,
  "eval_sampled_negatives": 255,
  "eval_batch_size": 128,
  "cleanup_every_rowgroups": 20,
  "zip_each_epoch": true,
  "item_vector_batch_size": 100000,
  "faiss_train_items": 500000,
  "faiss_nprobe": 64,
  "faiss_pq_m": 16,
  "faiss_pq_nbits": 8
}
Effective batch = 128


In [3]:
# Paths
WORK = Path("/kaggle/working")
ROOT = WORK / "merrec_models" / "GRU4Rec_FULL_CATALOG_V1"
for p in [
    ROOT,
    ROOT / "mappings",
    ROOT / "sequences",
    ROOT / "model",
    ROOT / "metrics",
    ROOT / "faiss",
]:
    p.mkdir(parents=True, exist_ok=True)

# EDIT these 3 paths to your MerRec prepared parquet files if needed.
# POP should contain item_id + interaction count on TRAIN.
# SEQ source should be event-level TRAIN/VAL/TEST data with user_id,item_id,event_id,stime.
DATA = Path("/kaggle/input")
print("Set your actual input paths below if auto-detection does not match.")


Set your actual input paths below if auto-detection does not match.


In [4]:
# Auto-discover MerRec parquet files by FULL PATH, not filename only.
# MerRec interaction files are commonly:
#   .../model_data/interactions/split=train/data_0.parquet
#   .../model_data/interactions/split=val/data_0.parquet   (if present)
#   .../model_data/interactions/split=test/data_0.parquet
# while popularity_train.parquet is NOT the interaction TRAIN file.

parquets = list(DATA.rglob("*.parquet"))
print("Found parquet files:", len(parquets))
for p in parquets[:100]:
    print(p)

def schema_of(path):
    return pl.scan_parquet(path).collect_schema().names()

def pick(cols, candidates):
    low = {c.lower(): c for c in cols}
    for x in candidates:
        if x.lower() in low:
            return low[x.lower()]
    raise KeyError(f"Cannot find any of {candidates} in columns={cols}")

def norm_path(p):
    return str(p).replace("\\", "/").lower()

def find_interaction_split(split_name):
    needle = f"/interactions/split={split_name.lower()}/"
    hits = [p for p in parquets if needle in norm_path(p)]
    if not hits:
        return None
    hits = sorted(hits, key=lambda x: norm_path(x))
    if len(hits) > 1:
        print(f"Found {len(hits)} files for split={split_name}; using first:", hits[0])
    return hits[0]

# Correct interaction split discovery.
TRAIN = find_interaction_split("train")
VAL_SPLIT = find_interaction_split("val")
TEST = find_interaction_split("test")

# Fallback validation file used by MerRec retrieval-training package.
eval_candidates = [
    p for p in parquets
    if "eval_events_with_cold_start.parquet" in norm_path(p)
]
VAL = VAL_SPLIT if VAL_SPLIT is not None else (eval_candidates[0] if eval_candidates else None)

# Popularity file must be detected separately.
pop_candidates = [
    p for p in parquets
    if p.name.lower() == "popularity_train.parquet"
]
POP = pop_candidates[0] if pop_candidates else None

print("\nResolved files:")
print("TRAIN =", TRAIN)
print("VAL   =", VAL)
print("TEST  =", TEST)
print("POP   =", POP)

if TRAIN is None:
    raise FileNotFoundError(
        "Could not find interaction TRAIN at .../interactions/split=train/*.parquet"
    )
if VAL is None:
    raise FileNotFoundError(
        "Could not find VAL split or eval_events_with_cold_start.parquet"
    )
if TEST is None:
    raise FileNotFoundError(
        "Could not find interaction TEST at .../interactions/split=test/*.parquet"
    )

# Guard against accidental popularity/eval assignment as TRAIN/TEST.
assert "/interactions/split=train/" in norm_path(TRAIN), f"Wrong TRAIN: {TRAIN}"
assert "/interactions/split=test/" in norm_path(TEST), f"Wrong TEST: {TEST}"
assert TRAIN != POP, "TRAIN must be interaction data, not popularity_train.parquet"
assert TEST != POP, "TEST must be interaction data, not popularity_train.parquet"

print("\n✅ Dataset path detection passed.")


Found parquet files: 11
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_data/eval_events_with_cold_start.parquet
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_data/popularity_train.parquet
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_data/cf_train.parquet
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_data/train_user_map.parquet
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_data/trending_train.parquet
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_data/train_item_map.parquet
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_data/item_catalog_train.parquet
/kaggle/input/datasets/ngytfd/merrec-retrieval-training/merrec_retrieval_training/model_specific/item_features.parquet
/kaggle/input/datasets/ngytfd/me

In [5]:
# Build FULL item vocabulary from TRAIN.
# Detect actual MerRec schema robustly.
ITEM_MAP = ROOT / "mappings" / "item_map.parquet"

train_cols = schema_of(TRAIN)
print("TRAIN columns =", train_cols)

TR_USER = pick(train_cols, ["user_id", "userid", "user"])
TR_ITEM = pick(train_cols, ["item_id", "itemid", "item", "product_id"])
TR_TIME = pick(train_cols, ["ts", "stime", "timestamp", "time", "event_time"])
TR_EVENT = pick(train_cols, ["event_id", "event", "behavior", "action", "event_group"])

print("Resolved TRAIN schema:")
print(" user  =", TR_USER)
print(" item  =", TR_ITEM)
print(" time  =", TR_TIME)
print(" event =", TR_EVENT)

# Prefer provided popularity_train.parquet for full train catalog/popularity.
if POP is not None:
    pop_cols = schema_of(POP)
    print("POP columns =", pop_cols)

    POP_ITEM = pick(pop_cols, ["item_id", "itemid", "item", "product_id"])
    POP_FREQ = pick(pop_cols, ["interactions", "count", "n_interactions", "frequency", "freq"])

    item_map = (
        pl.scan_parquet(POP)
        .select([
            pl.col(POP_ITEM).cast(pl.Utf8).alias("item_id"),
            pl.col(POP_FREQ).cast(pl.UInt64).fill_null(0).alias("interactions"),
        ])
        .collect(engine="streaming")
        .unique(subset=["item_id"], keep="first")
        .sort(["interactions", "item_id"], descending=[True, False])
        .with_row_index("item_idx", offset=2)
        .with_columns(pl.col("item_idx").cast(pl.UInt32))
        .select(["item_id", "item_idx", "interactions"])
    )
else:
    item_map = (
        pl.scan_parquet(TRAIN)
        .select(pl.col(TR_ITEM).cast(pl.Utf8).alias("item_id"))
        .group_by("item_id")
        .agg(pl.len().cast(pl.UInt64).alias("interactions"))
        .collect(engine="streaming")
        .sort(["interactions", "item_id"], descending=[True, False])
        .with_row_index("item_idx", offset=2)
        .with_columns(pl.col("item_idx").cast(pl.UInt32))
        .select(["item_id", "item_idx", "interactions"])
    )

N_ITEMS = item_map.height
VOCAB_SIZE = N_ITEMS + 2

if N_ITEMS < 2:
    raise RuntimeError("Catalog too small.")

assert int(item_map["item_idx"].min()) == 2
assert int(item_map["item_idx"].max()) == VOCAB_SIZE - 1
assert int(item_map["item_idx"].n_unique()) == N_ITEMS

item_map.write_parquet(ITEM_MAP, compression="zstd")

print(f"FULL catalog items = {N_ITEMS:,}")
print(f"VOCAB_SIZE         = {VOCAB_SIZE:,}")

emb_bytes = VOCAB_SIZE * CFG["item_embed_dim"] * 4
print(f"Item embedding FP32 size ≈ {emb_bytes/2**30:.2f} GB")


TRAIN columns = ['user_id', 'item_id', 'product_id', 'session_id', 'sequence_id', 'event_id', 'event_group', 'ts', 'price', 'name', 'category0', 'category1', 'category2', 'brand', 'condition', 'shipper', 'event_weight', 'is_strong_positive', 'is_purchase']
Resolved TRAIN schema:
 user  = user_id
 item  = item_id
 time  = ts
 event = event_id
POP columns = ['item_id', 'popularity_score', 'interactions', 'unique_users', 'views', 'likes', 'carts', 'offers', 'buy_starts', 'purchases', 'last_event_ts']
FULL catalog items = 27,328,461
VOCAB_SIZE         = 27,328,463
Item embedding FP32 size ≈ 3.26 GB


In [6]:
# Build id lookup dictionary.
# For very large catalogs this dictionary itself consumes RAM, but is still much smaller than GPU full embedding.
item_to_idx = dict(zip(item_map["item_id"].to_list(), item_map["item_idx"].to_list()))
print("item_to_idx entries =", len(item_to_idx))

def event_to_idx(x):
    if x is None:
        return EVENT_TO_IDX["unknown"]
    s = str(x).lower()
    # normalize common MerRec names
    if "view" in s:
        return EVENT_TO_IDX["view"]
    if "like" in s:
        return EVENT_TO_IDX["like"]
    if "cart" in s:
        return EVENT_TO_IDX["cart"]
    if "offer" in s:
        return EVENT_TO_IDX["offer"]
    if "buy" in s and ("comp" in s or "complete" in s):
        return EVENT_TO_IDX["buy_comp"]
    if "buy" in s:
        return EVENT_TO_IDX["buy_start"]
    return EVENT_TO_IDX["unknown"]


item_to_idx entries = 27328461


In [7]:
# Build temporal sequence parquet from TRAIN.
SEQ_FILE = ROOT / "sequences" / "train_sequences.parquet"

def build_sequences(src, out_path, max_seq_len):
    cols = schema_of(src)
    U = pick(cols, ["user_id", "userid", "user"])
    I = pick(cols, ["item_id", "itemid", "item", "product_id"])
    T = pick(cols, ["ts", "stime", "timestamp", "time", "event_time"])
    E = pick(cols, ["event_id", "event", "behavior", "action", "event_group"])

    print("Building sequences from schema:")
    print(" user =", U, "| item =", I, "| time =", T, "| event =", E)

    df = (
        pl.scan_parquet(src)
        .select([
            pl.col(U).cast(pl.Utf8).alias("user_id"),
            pl.col(I).cast(pl.Utf8).alias("item_id"),
            pl.col(E).cast(pl.Utf8).alias("event_id"),
            pl.col(T).cast(pl.Int64, strict=False).alias("ts"),
        ])
        .filter(pl.col("user_id").is_not_null() & pl.col("item_id").is_not_null())
        .collect(engine="streaming")
        .sort(["user_id", "ts"])
    )

    rows = []
    for user, sub in df.group_by("user_id", maintain_order=True):
        if isinstance(user, tuple):
            user = user[0]

        its = []
        evs = []

        for item_id, event_id in sub.select(["item_id", "event_id"]).iter_rows():
            idx = item_to_idx.get(str(item_id), OOV)
            its.append(int(idx))
            evs.append(int(event_to_idx(event_id)))

        if len(its) >= 2:
            # Need max_seq_len inputs + one next target.
            its = its[-(max_seq_len + 1):]
            evs = evs[-(max_seq_len + 1):]
            rows.append((str(user), its, evs))

    if not rows:
        raise RuntimeError("No valid training sequences were built.")

    out = pl.DataFrame(
        rows,
        schema=["user_id", "seq_items", "seq_events"],
        orient="row",
    )
    out.write_parquet(out_path, compression="zstd", row_group_size=10000)

    lengths = out["seq_items"].list.len()
    print("Sequences:", out.height, "->", out_path)
    print(
        "Sequence length stats:",
        "min=", int(lengths.min()),
        "mean=", float(lengths.mean()),
        "max=", int(lengths.max()),
    )

build_sequences(TRAIN, SEQ_FILE, CFG["max_seq_len"])


Building sequences from schema:
 user = user_id | item = item_id | time = ts | event = event_id
Sequences: 2225414 -> /kaggle/working/merrec_models/GRU4Rec_FULL_CATALOG_V1/sequences/train_sequences.parquet
Sequence length stats: min= 2 mean= 33.40681284471114 max= 101


In [8]:
# Build one-step validation/test target bundles.
# NOTE: true query history is built separately from chronological interaction history.
def build_eval_bundle(path, max_users):
    cols = schema_of(path)
    print(f"EVAL schema for {path.name} =", cols)

    U = pick(cols, ["user_id", "userid", "user"])
    I = pick(cols, ["item_id", "itemid", "item", "product_id"])
    T = pick(cols, ["ts", "stime", "timestamp", "time", "event_time"])
    E = pick(cols, ["event_id", "event", "behavior", "action", "event_group"])

    df = (
        pl.scan_parquet(path)
        .select([
            pl.col(U).cast(pl.Utf8).alias("user_id"),
            pl.col(I).cast(pl.Utf8).alias("item_id"),
            pl.col(E).cast(pl.Utf8).alias("event_id"),
            pl.col(T).cast(pl.Int64, strict=False).alias("ts"),
        ])
        .filter(pl.col("user_id").is_not_null() & pl.col("item_id").is_not_null())
        .collect(engine="streaming")
        .sort(["user_id", "ts"])
    )

    users, targets, target_events, target_ts = [], [], [], []

    for user, sub in df.group_by("user_id", maintain_order=True):
        if isinstance(user, tuple):
            user = user[0]

        first_item = str(sub["item_id"][0])
        target = item_to_idx.get(first_item)

        # Full catalog is based on TRAIN vocabulary.
        # Cold-start unseen items cannot have a learned item-id embedding.
        if target is not None:
            users.append(str(user))
            targets.append(int(target))
            target_events.append(str(sub["event_id"][0]))
            target_ts.append(int(sub["ts"][0]) if sub["ts"][0] is not None else None)

        if max_users is not None and len(users) >= max_users:
            break

    return {
        "users": users,
        "targets": targets,
        "target_events": target_events,
        "target_ts": target_ts,
    }

VAL_BUNDLE = build_eval_bundle(VAL, CFG["val_users"])
TEST_BUNDLE = build_eval_bundle(TEST, CFG["test_users"])

print("VAL evaluable users :", len(VAL_BUNDLE["users"]))
print("TEST evaluable users:", len(TEST_BUNDLE["users"]))


EVAL schema for data_0.parquet = ['user_id', 'item_id', 'product_id', 'session_id', 'sequence_id', 'event_id', 'event_group', 'ts', 'price', 'name', 'category0', 'category1', 'category2', 'brand', 'condition', 'shipper', 'event_weight', 'is_strong_positive', 'is_purchase']
EVAL schema for data_0.parquet = ['user_id', 'item_id', 'product_id', 'session_id', 'sequence_id', 'event_id', 'event_group', 'ts', 'price', 'name', 'category0', 'category1', 'category2', 'brand', 'condition', 'shipper', 'event_weight', 'is_strong_positive', 'is_purchase']
VAL evaluable users : 10000
TEST evaluable users: 10000


In [9]:
# Build TRAIN histories for sampled validation.
# Query history của VAL user được lấy từ TRAIN trước target VAL đầu tiên.
# Đây là history đúng chiều thời gian cho one-step next-item validation.

def build_train_histories_for_eval(eval_bundle, max_users=None):
    users = eval_bundle["users"]
    targets = eval_bundle["targets"]

    if max_users is not None:
        users = users[:max_users]
        targets = targets[:max_users]

    target_by_user = dict(zip(users, targets))
    user_set = list(target_by_user.keys())

    cols = schema_of(TRAIN)
    U = pick(cols, ["user_id", "userid", "user"])
    I = pick(cols, ["item_id", "itemid", "item", "product_id"])
    T = pick(cols, ["ts", "stime", "timestamp", "time", "event_time"])
    E = pick(cols, ["event_id", "event", "behavior", "action", "event_group"])

    df = (
        pl.scan_parquet(TRAIN)
        .select([
            pl.col(U).cast(pl.Utf8).alias("user_id"),
            pl.col(I).cast(pl.Utf8).alias("item_id"),
            pl.col(E).cast(pl.Utf8).alias("event_id"),
            pl.col(T).cast(pl.Int64, strict=False).alias("ts"),
        ])
        .filter(pl.col("user_id").is_in(user_set))
        .filter(pl.col("item_id").is_not_null())
        .collect(engine="streaming")
        .sort(["user_id", "ts"])
    )

    rows = []
    for user, sub in df.group_by("user_id", maintain_order=True):
        if isinstance(user, tuple):
            user = user[0]
        user = str(user)

        if user not in target_by_user:
            continue

        raw_items = sub["item_id"].to_list()
        raw_events = sub["event_id"].to_list()

        mapped_items = [int(item_to_idx.get(str(x), OOV)) for x in raw_items]
        mapped_events = [int(event_to_idx(x)) for x in raw_events]

        # Remove padding-like invalid history items and keep recent history only.
        hist = [
            (i, e) for i, e in zip(mapped_items, mapped_events)
            if 0 <= i < VOCAB_SIZE
        ][-CFG["max_seq_len"]:]

        if not hist:
            continue

        rows.append({
            "user_id": user,
            "items": [x[0] for x in hist],
            "events": [x[1] for x in hist],
            "target": int(target_by_user[user]),
        })

    print("Sampled VAL histories:", len(rows))
    return rows

VAL_EVAL_ROWS = build_train_histories_for_eval(
    VAL_BUNDLE,
    max_users=CFG["eval_sampled_users"],
)


Sampled VAL histories: 1810


In [10]:
# Model: item embedding on CPU, dense sequence encoder on GPU.
class GRU4RecFullCatalog(nn.Module):
    def __init__(
        self, vocab_size, event_vocab_size,
        item_embed_dim, event_embed_dim, hidden_dim,
        gru_layers, dropout
    ):
        super().__init__()

        # CPU sparse embedding
        self.item_emb = nn.Embedding(
            vocab_size,
            item_embed_dim,
            padding_idx=PAD,
            sparse=True,
            device="cpu",
        )

        # GPU modules
        self.event_emb = nn.Embedding(event_vocab_size, event_embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(item_embed_dim + event_embed_dim, hidden_dim)
        self.input_ln = nn.LayerNorm(hidden_dim)
        self.gru = nn.GRU(
            hidden_dim,
            hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0,
        )
        self.out_ln = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.item_out = nn.Linear(item_embed_dim, hidden_dim, bias=False)

        nn.init.normal_(self.item_emb.weight, std=0.02)
        with torch.no_grad():
            self.item_emb.weight[PAD].zero_()

    def encode_sequence(self, item_ids_cpu, event_ids_gpu, return_all=False):
        # gather CPU item embeddings then move just the batch to GPU
        item_x = self.item_emb(item_ids_cpu).to(DEVICE, non_blocking=True)
        event_x = self.event_emb(event_ids_gpu)
        x = torch.cat([item_x, event_x], dim=-1)
        x = self.input_proj(x)
        x = self.input_ln(x)
        x = self.dropout(x)
        out, _ = self.gru(x)
        out = self.out_ln(out)
        if return_all:
            return out

        lengths = item_ids_cpu.ne(PAD).sum(dim=1).clamp_min(1).to(DEVICE)
        return out[
            torch.arange(out.size(0), device=DEVICE),
            lengths - 1
        ]

    def item_vectors(self, ids_cpu):
        v = self.item_emb(ids_cpu).to(DEVICE, non_blocking=True)
        v = self.item_out(v)
        return F.normalize(v, dim=-1)

model = GRU4RecFullCatalog(
    VOCAB_SIZE,
    EVENT_VOCAB_SIZE,
    CFG["item_embed_dim"],
    CFG["event_embed_dim"],
    CFG["hidden_dim"],
    CFG["gru_layers"],
    CFG["dropout"],
)

# Move only dense modules to GPU
model.event_emb = model.event_emb.to(DEVICE)
model.input_proj = model.input_proj.to(DEVICE)
model.input_ln = model.input_ln.to(DEVICE)
model.gru = model.gru.to(DEVICE)
model.out_ln = model.out_ln.to(DEVICE)
model.dropout = model.dropout.to(DEVICE)
model.item_out = model.item_out.to(DEVICE)

print(model)
print("item_emb device:", model.item_emb.weight.device)
print("gru device:", next(model.gru.parameters()).device)


GRU4RecFullCatalog(
  (item_emb): Embedding(27328463, 32, padding_idx=0, sparse=True)
  (event_emb): Embedding(7, 16, padding_idx=0)
  (input_proj): Linear(in_features=48, out_features=128, bias=True)
  (input_ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (gru): GRU(128, 128, num_layers=2, batch_first=True, dropout=0.2)
  (out_ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (item_out): Linear(in_features=32, out_features=128, bias=False)
)
item_emb device: cpu
gru device: cuda:0


In [11]:
# Batch + full-catalog negative sampling.
def rows_to_batch(items_list, events_list):
    B = len(items_list)
    T = max(len(x) - 1 for x in items_list)

    inp_i = torch.full((B, T), PAD, dtype=torch.long, device="cpu")
    inp_e = torch.zeros((B, T), dtype=torch.long, device=DEVICE)
    tgt = torch.full((B, T), -100, dtype=torch.long, device="cpu")

    for b, (its, evs) in enumerate(zip(items_list, events_list)):
        n = len(its) - 1
        if n <= 0:
            continue
        x = torch.as_tensor(its[:-1], dtype=torch.long)
        e = torch.as_tensor(evs[:-1], dtype=torch.long)
        y = torch.as_tensor(its[1:], dtype=torch.long)

        x[(x < 0) | (x >= VOCAB_SIZE)] = OOV
        y[(y <= OOV) | (y >= VOCAB_SIZE)] = -100
        e[(e < 0) | (e >= EVENT_VOCAB_SIZE)] = EVENT_TO_IDX["unknown"]

        inp_i[b, :n] = x
        inp_e[b, :n] = e.to(DEVICE)
        tgt[b, :n] = y

    return inp_i, inp_e, tgt

def sample_negatives_cpu(n_rows, n_neg):
    return torch.randint(
        low=2,
        high=VOCAB_SIZE,
        size=(int(n_rows), int(n_neg)),
        dtype=torch.long,
        device="cpu",
    )

def sampled_ce_loss(hidden_gpu, targets_cpu):
    mask_cpu = (targets_cpu > OOV) & (targets_cpu < VOCAB_SIZE)
    if not mask_cpu.any():
        return None, 0

    mask_gpu = mask_cpu.to(DEVICE)
    h = F.normalize(hidden_gpu[mask_gpu], dim=-1)
    pos_cpu = targets_cpu[mask_cpu].long()

    neg_cpu = sample_negatives_cpu(pos_cpu.numel(), CFG["num_negatives"])

    # avoid positive collisions
    for _ in range(8):
        clash = neg_cpu.eq(pos_cpu[:, None])
        if not clash.any():
            break
        neg_cpu[clash] = torch.randint(
            2, VOCAB_SIZE,
            (int(clash.sum().item()),),
            dtype=torch.long,
        )

    pos_v = model.item_vectors(pos_cpu)
    neg_v = model.item_vectors(neg_cpu)

    pos_logit = (h * pos_v).sum(-1, keepdim=True)
    neg_logit = torch.einsum("bd,bnd->bn", h, neg_v)
    logits = torch.cat([pos_logit, neg_logit], dim=1) / CFG["temperature"]

    labels = torch.zeros(pos_cpu.size(0), dtype=torch.long, device=DEVICE)
    loss = F.cross_entropy(logits, labels)
    return loss, int(pos_cpu.size(0))


In [12]:
# Optimizers
dense_params = []
for name, p in model.named_parameters():
    if name != "item_emb.weight":
        dense_params.append(p)

dense_optimizer = torch.optim.AdamW(
    dense_params,
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)
item_optimizer = torch.optim.SGD(
    [model.item_emb.weight],
    lr=CFG["item_lr"],
)

AMP_ENABLED = bool(CFG["amp"] and DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED) if DEVICE.type == "cuda" else None

def zero_all():
    dense_optimizer.zero_grad(set_to_none=True)
    item_optimizer.zero_grad(set_to_none=True)

def step_all():
    if AMP_ENABLED:
        scaler.unscale_(dense_optimizer)
    torch.nn.utils.clip_grad_norm_(dense_params, CFG["grad_clip"])

    if AMP_ENABLED:
        scaler.step(dense_optimizer)
        scaler.step(item_optimizer)
        scaler.update()
    else:
        dense_optimizer.step()
        item_optimizer.step()

    zero_all()


In [13]:
# Fast sampled validation after each epoch.
# NOTE:
# - Metric này dùng target thật của VAL, history lấy từ TRAIN.
# - Mỗi target được rank cùng N sampled negatives lấy từ toàn bộ TRAIN catalog.
# - Vì vậy đây là "sampled Recall/NDCG/MRR", KHÔNG phải full-catalog metric.
# - Full-catalog evaluation vẫn nên chạy cuối cùng bằng FAISS.

@torch.no_grad()
def evaluate_sampled_val(model, rows, ks=None):
    if ks is None:
        ks = CFG["eval_ks"]

    if not rows:
        return {}

    model.eval()
    max_k = max(ks)
    n_neg = int(CFG["eval_sampled_negatives"])
    bs = int(CFG["eval_batch_size"])

    sums = {}
    n_users = 0
    t0 = time.time()

    for st in range(0, len(rows), bs):
        chunk = rows[st:st + bs]
        B = len(chunk)
        if B == 0:
            continue

        lengths = [len(r["items"]) for r in chunk]
        T = max(lengths)

        item_cpu = torch.full((B, T), PAD, dtype=torch.long, device="cpu")
        event_gpu = torch.zeros((B, T), dtype=torch.long, device=DEVICE)

        for b, r in enumerate(chunk):
            n = len(r["items"])
            item_cpu[b, :n] = torch.as_tensor(r["items"], dtype=torch.long)
            event_gpu[b, :n] = torch.as_tensor(r["events"], dtype=torch.long, device=DEVICE)

        q = model.encode_sequence(item_cpu, event_gpu, return_all=False)
        q = F.normalize(q, dim=-1)

        pos_cpu = torch.as_tensor(
            [r["target"] for r in chunk],
            dtype=torch.long,
            device="cpu",
        )

        neg_cpu = torch.randint(
            2, VOCAB_SIZE,
            (B, n_neg),
            dtype=torch.long,
            device="cpu",
        )

        # Avoid positive collision.
        for _ in range(4):
            clash = neg_cpu.eq(pos_cpu[:, None])
            if not clash.any():
                break
            neg_cpu[clash] = torch.randint(
                2,
                VOCAB_SIZE,
                (int(clash.sum().item()),),
                dtype=torch.long,
            )

        pos_v = model.item_vectors(pos_cpu)
        neg_v = model.item_vectors(neg_cpu)

        pos_score = (q * pos_v).sum(-1)
        neg_score = torch.einsum("bd,bnd->bn", q, neg_v)

        # Rank 1 = best.
        ranks = 1 + (neg_score > pos_score[:, None]).sum(dim=1)
        ranks = ranks.detach().cpu().numpy()

        for rank in ranks:
            rank = int(rank)
            for k in ks:
                hit = 1.0 if rank <= k else 0.0
                sums[f"Recall@{k}"] = sums.get(f"Recall@{k}", 0.0) + hit
                sums[f"NDCG@{k}"] = sums.get(f"NDCG@{k}", 0.0) + (
                    1.0 / math.log2(rank + 1) if rank <= k else 0.0
                )
                sums[f"MRR@{k}"] = sums.get(f"MRR@{k}", 0.0) + (
                    1.0 / rank if rank <= k else 0.0
                )

        n_users += B

        del item_cpu, event_gpu, q, pos_cpu, neg_cpu, pos_v, neg_v, pos_score, neg_score

    model.train()

    out = {k: v / max(1, n_users) for k, v in sums.items()}
    out["eval_users"] = int(n_users)
    out["eval_negatives"] = int(n_neg)
    out["eval_seconds"] = float(time.time() - t0)
    return out


In [14]:
# Smoke test: forward + backward before long training.
pf = pq.ParquetFile(SEQ_FILE)
tab = pf.read_row_group(0, columns=["seq_items", "seq_events"])
items = tab.column("seq_items").to_pylist()[:8]
events = tab.column("seq_events").to_pylist()[:8]

xi_cpu, xe_gpu, y_cpu = rows_to_batch(items, events)

zero_all()
with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
    h = model.encode_sequence(xi_cpu, xe_gpu, return_all=True)
    loss, count = sampled_ce_loss(h, y_cpu)

if loss is None:
    raise RuntimeError("Smoke batch has no valid targets.")

if AMP_ENABLED:
    scaler.scale(loss).backward()
else:
    loss.backward()

print("✅ SMOKE TEST PASSED")
print("loss =", float(loss.detach()), "| targets =", count)

zero_all()
del tab, items, events, xi_cpu, xe_gpu, y_cpu, h, loss
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


✅ SMOKE TEST PASSED
loss = 4.458039283752441 | targets = 155


### Auto-save checkpoint sau mỗi epoch

- Mỗi epoch hoàn thành sẽ cập nhật `gru4rec_full_catalog_latest.pt`.
- Đồng thời tạo/ghi đè `/kaggle/working/GRU4Rec_FULL_CATALOG_LATEST.zip`.
- Dùng file tạm + `os.replace()` để hạn chế checkpoint/ZIP bị hỏng nếu phiên dừng giữa lúc ghi.
- ZIP dùng chế độ không nén (`ZIP_STORED`) vì checkpoint embedding rất lớn; mục tiêu là đóng gói nhanh, không phải giảm dung lượng.


In [15]:
# Training FAST + per-epoch sampled metrics + AUTO-SAVE + ZIP overwrite
# Tối ưu chính:
# - batch_size lớn hơn, grad_accum nhỏ hơn
# - 64 sampled negatives thay vì 128
# - tối đa 1M windows/epoch thay vì 2M
# - không gc.collect()/empty_cache sau mọi row-group
# - đánh giá sampled VAL sau mỗi epoch
# - checkpoint + ZIP luôn ghi đè bản epoch trước

import os
import zipfile

CHECKPOINT = ROOT / "model" / "gru4rec_full_catalog_latest.pt"
CHECKPOINT_TMP = ROOT / "model" / "gru4rec_full_catalog_latest.tmp"

BEST_CHECKPOINT = ROOT / "model" / "gru4rec_full_catalog_best_sampled.pt"
BEST_TMP = ROOT / "model" / "gru4rec_full_catalog_best_sampled.tmp"

MODEL_ZIP = WORK / "GRU4Rec_FULL_CATALOG_LATEST.zip"
MODEL_ZIP_TMP = WORK / "GRU4Rec_FULL_CATALOG_LATEST.tmp.zip"

HISTORY_JSON = ROOT / "metrics" / "train_history.json"

HISTORY = []
best_score = -1.0
best_epoch = 0
bad_epochs = 0

ACCUM = CFG["grad_accum_steps"]
MAX_WINDOWS = CFG["max_train_windows_per_epoch"]
CLEANUP_EVERY = max(1, int(CFG["cleanup_every_rowgroups"]))

for ep in range(1, CFG["epochs"] + 1):
    model.train()

    loss_sum = 0.0
    batches = 0
    targets = 0
    windows = 0
    micro = 0
    t0 = time.time()

    zero_all()

    rg_order = list(range(pf.num_row_groups))
    random.Random(SEED + ep).shuffle(rg_order)

    stop = False

    for rg_pos, rg in enumerate(rg_order, start=1):
        tab = pf.read_row_group(rg, columns=["seq_items", "seq_events"])
        items = tab.column("seq_items").to_pylist()
        events = tab.column("seq_events").to_pylist()

        order = np.random.permutation(len(items))

        for st in range(0, len(order), CFG["batch_size"]):
            if MAX_WINDOWS is not None and windows >= MAX_WINDOWS:
                stop = True
                break

            ix = order[st: st + CFG["batch_size"]]

            if MAX_WINDOWS is not None:
                ix = ix[: MAX_WINDOWS - windows]

            if len(ix) == 0:
                stop = True
                break

            bi = [items[j] for j in ix]
            be = [events[j] for j in ix]
            windows += len(ix)

            xi_cpu, xe_gpu, y_cpu = rows_to_batch(bi, be)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                h = model.encode_sequence(
                    xi_cpu,
                    xe_gpu,
                    return_all=True,
                )
                loss, count = sampled_ce_loss(h, y_cpu)

            if loss is None:
                continue

            scaled = loss / ACCUM

            if AMP_ENABLED:
                scaler.scale(scaled).backward()
            else:
                scaled.backward()

            micro += 1

            if micro % ACCUM == 0:
                step_all()

            loss_sum += float(loss.detach())
            batches += 1
            targets += count

            del xi_cpu, xe_gpu, y_cpu, h, loss

        del tab, items, events, order

        # Cleanup thưa hơn để tránh mất thời gian đồng bộ CPU/GPU.
        if rg_pos % CLEANUP_EVERY == 0:
            gc.collect()

        if stop:
            break

    if micro % ACCUM != 0:
        step_all()

    train_seconds = time.time() - t0

    rec = {
        "epoch": ep,
        "windows": int(windows),
        "batches": int(batches),
        "targets": int(targets),
        "train_loss": float(loss_sum / max(1, batches)),
        "train_seconds": float(train_seconds),
    }

    # =========================================================
    # SAMPLED VALIDATION MỖI EPOCH
    # =========================================================
    if ep % CFG["eval_every"] == 0:
        val_metrics = evaluate_sampled_val(
            model,
            VAL_EVAL_ROWS,
            ks=CFG["eval_ks"],
        )

        for k, v in val_metrics.items():
            rec[f"val_{k}"] = v

        print("\n" + "=" * 78)
        print(
            f"EPOCH {ep} | "
            f"loss={rec['train_loss']:.6f} | "
            f"windows={rec['windows']:,} | "
            f"train={rec['train_seconds']/60:.1f} min"
        )

        for k in CFG["eval_ks"]:
            print(
                f"VAL sampled @{k:>3}: "
                f"Recall={val_metrics.get(f'Recall@{k}', 0):.6f} | "
                f"NDCG={val_metrics.get(f'NDCG@{k}', 0):.6f} | "
                f"MRR={val_metrics.get(f'MRR@{k}', 0):.6f}"
            )

        print(
            f"VAL users={val_metrics.get('eval_users', 0):,} | "
            f"negatives/user={val_metrics.get('eval_negatives', 0)} | "
            f"eval={val_metrics.get('eval_seconds', 0):.1f}s"
        )
        print("=" * 78 + "\n")
    else:
        print(json.dumps(rec, indent=2))

    HISTORY.append(rec)

    # =========================================================
    # SAVE LATEST CHECKPOINT (epoch mới ghi đè epoch trước)
    # =========================================================
    checkpoint = {
        "model": model.state_dict(),
        "dense_optimizer": dense_optimizer.state_dict(),
        "item_optimizer": item_optimizer.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "cfg": CFG,
        "vocab_size": VOCAB_SIZE,
        "epoch": ep,
        "history": HISTORY,
        "best_sampled_recall20": best_score,
        "best_epoch": best_epoch,
    }

    torch.save(checkpoint, CHECKPOINT_TMP)
    os.replace(CHECKPOINT_TMP, CHECKPOINT)

    # =========================================================
    # OPTIONAL BEST MODEL BY SAMPLED Recall@20
    # =========================================================
    score = rec.get("val_Recall@20")

    if score is not None and score > best_score:
        best_score = float(score)
        best_epoch = ep
        bad_epochs = 0

        checkpoint["best_sampled_recall20"] = best_score
        checkpoint["best_epoch"] = best_epoch

        torch.save(checkpoint, BEST_TMP)
        os.replace(BEST_TMP, BEST_CHECKPOINT)

        print(
            f"🏆 New best sampled Recall@20={best_score:.6f} "
            f"at epoch {best_epoch}"
        )
    elif score is not None:
        bad_epochs += 1

    with open(HISTORY_JSON, "w", encoding="utf-8") as f:
        json.dump(HISTORY, f, ensure_ascii=False, indent=2)

    # =========================================================
    # ZIP LATEST MODEL (ghi đè ZIP cũ)
    # =========================================================
    if CFG["zip_each_epoch"]:
        with zipfile.ZipFile(
            MODEL_ZIP_TMP,
            mode="w",
            compression=zipfile.ZIP_STORED,
            allowZip64=True,
        ) as zf:
            zf.write(CHECKPOINT, arcname=CHECKPOINT.name)
            zf.write(HISTORY_JSON, arcname="train_history.json")

        os.replace(MODEL_ZIP_TMP, MODEL_ZIP)

    print("-" * 78)
    print(f"✅ SAVED EPOCH {ep}")
    print(f"Latest checkpoint : {CHECKPOINT}")
    print(f"Best checkpoint   : {BEST_CHECKPOINT}")
    if CFG["zip_each_epoch"]:
        print(f"Latest ZIP        : {MODEL_ZIP}")
        print(f"ZIP size          : {MODEL_ZIP.stat().st_size / (1024**3):.2f} GB")
    print("-" * 78)

print("\nTraining finished.")
print("Latest checkpoint:", CHECKPOINT)
print("Best checkpoint  :", BEST_CHECKPOINT)
print("Latest ZIP       :", MODEL_ZIP)



EPOCH 1 | loss=3.994050 | windows=1,000,000 | train=26.9 min
VAL sampled @ 10: Recall=0.091713 | NDCG=0.053663 | MRR=0.042259
VAL sampled @ 20: Recall=0.144199 | NDCG=0.066778 | MRR=0.045776
VAL sampled @ 50: Recall=0.260221 | NDCG=0.089399 | MRR=0.049248
VAL sampled @100: Recall=0.443094 | NDCG=0.118843 | MRR=0.051780
VAL users=1,810 | negatives/user=255 | eval=0.3s

🏆 New best sampled Recall@20=0.144199 at epoch 1
------------------------------------------------------------------------------
✅ SAVED EPOCH 1
Latest checkpoint : /kaggle/working/merrec_models/GRU4Rec_FULL_CATALOG_V1/model/gru4rec_full_catalog_latest.pt
Best checkpoint   : /kaggle/working/merrec_models/GRU4Rec_FULL_CATALOG_V1/model/gru4rec_full_catalog_best_sampled.pt
Latest ZIP        : /kaggle/working/GRU4Rec_FULL_CATALOG_LATEST.zip
ZIP size          : 3.26 GB
------------------------------------------------------------------------------

EPOCH 2 | loss=3.956457 | windows=1,000,000 | train=24.4 min
VAL sampled @ 10: R

In [16]:
# Build full-catalog FAISS index in chunks.
if faiss is None:
    raise ImportError("faiss is not installed.")

model.eval()
DIM = CFG["hidden_dim"]

# IVF-PQ is appropriate for multi-million catalog.
nlist = int(4 * math.sqrt(N_ITEMS))
nlist = max(4096, min(nlist, 65536))

quantizer = faiss.IndexFlatIP(DIM)
index = faiss.IndexIVFPQ(
    quantizer,
    DIM,
    nlist,
    CFG["faiss_pq_m"],
    CFG["faiss_pq_nbits"],
    faiss.METRIC_INNER_PRODUCT,
)

# Train FAISS on sampled catalog vectors.
rng = np.random.default_rng(SEED)
sample_n = min(CFG["faiss_train_items"], N_ITEMS)
sample_ids = rng.choice(
    np.arange(2, VOCAB_SIZE, dtype=np.int64),
    size=sample_n,
    replace=False,
)

train_vecs = []
BS = CFG["item_vector_batch_size"]

with torch.no_grad():
    for st in range(0, sample_n, BS):
        ids = torch.from_numpy(sample_ids[st:st+BS]).long()
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            v = model.item_vectors(ids).float().cpu().numpy()
        train_vecs.append(v)

train_vecs = np.concatenate(train_vecs, axis=0).astype("float32")
index.train(train_vecs)
del train_vecs

# Add all catalog vectors with their real item_idx as FAISS ids.
index = faiss.IndexIDMap2(index)

with torch.no_grad():
    for start in range(2, VOCAB_SIZE, BS):
        end = min(start + BS, VOCAB_SIZE)
        ids_np = np.arange(start, end, dtype=np.int64)
        ids = torch.from_numpy(ids_np).long()

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            v = model.item_vectors(ids).float().cpu().numpy().astype("float32")

        index.add_with_ids(v, ids_np)

        if start % (BS * 20) == 2:
            print(f"indexed {end-2:,}/{N_ITEMS:,}")

index.index.nprobe = CFG["faiss_nprobe"]
faiss.write_index(index, str(ROOT / "faiss" / "full_catalog.index"))
print("FAISS ntotal =", index.ntotal)


ImportError: faiss is not installed.

In [ ]:
# Metrics helpers
def metrics_at_k(ranked_ids, target, ks):
    out = {}
    for k in ks:
        top = ranked_ids[:k]
        hit = target in top
        out[f"Recall@{k}"] = 1.0 if hit else 0.0
        if hit:
            rank = top.index(target) + 1
            out[f"NDCG@{k}"] = 1.0 / math.log2(rank + 1)
            out[f"MRR@{k}"] = 1.0 / rank
        else:
            out[f"NDCG@{k}"] = 0.0
            out[f"MRR@{k}"] = 0.0
    return out

def aggregate_metrics(rows):
    if not rows:
        return {}
    keys = rows[0].keys()
    return {k: float(np.mean([r[k] for r in rows])) for k in keys}


In [ ]:
# IMPORTANT:
# To evaluate true next-item, the query history for each VAL/TEST user must be built
# from TRAIN history (and TRAIN+VAL history for TEST), not from the target split alone.
# Implement this after confirming your exact MerRec split files/schema.
#
# This notebook intentionally does NOT fabricate history that isn't present in the supplied split files.
print("Training + full-catalog indexing pipeline is complete.")
print("Next step: build exact temporal eval histories from your TRAIN/VAL/TEST split convention.")


In [ ]:
# Serving helper using an explicit user history.
idx_to_item = dict(zip(item_map["item_idx"].to_list(), item_map["item_id"].to_list()))

@torch.no_grad()
def recommend_next(history_item_ids, history_events, topk=20):
    # history_item_ids: original raw item_id values
    # history_events: behavior strings
    mapped = [int(item_to_idx.get(str(x), OOV)) for x in history_item_ids]
    evs = [int(event_to_idx(x)) for x in history_events]

    mapped = mapped[-CFG["max_seq_len"]:]
    evs = evs[-CFG["max_seq_len"]:]

    if len(mapped) == 0:
        raise ValueError("history is empty")

    xi_cpu = torch.tensor([mapped], dtype=torch.long, device="cpu")
    xe_gpu = torch.tensor([evs], dtype=torch.long, device=DEVICE)

    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
        q = model.encode_sequence(xi_cpu, xe_gpu, return_all=False)

    q = F.normalize(q, dim=-1).float().cpu().numpy().astype("float32")
    scores, ids = index.search(q, topk)

    result = []
    for item_idx, score in zip(ids[0], scores[0]):
        if item_idx < 2:
            continue
        result.append({
            "item_idx": int(item_idx),
            "item_id": idx_to_item.get(int(item_idx)),
            "score": float(score),
        })
    return result

print("recommend_next() ready.")


# EXPORT CANDIDATES FOR DCN RANKER — GRU4REC

Bổ sung bước Export Top-K Candidates cho 3 split TRAIN, VAL, TEST vào `D:\MerRec\candidate_exports`.
Mỗi split sẽ tạo file parquet có schema chuẩn:
`case_id`, `user_id`, `candidate_item_id`, `target_item_id`, `score`, `rank`

In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"D:\MerRec")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.export_all_candidate_datasets import export_gru

print("=== EXPORTING GRU4REC CANDIDATES FOR DCN RANKER ===")
for split in ["train", "val", "test"]:
    export_gru(split)
